# Customer Churn Analysis

## Exploratory Data Analysis

This project analyzes telecom customer churn to identify customer segments, contract characteristics, and service attributes associated with higher churn.

### Business Objectives

- Measure the overall customer churn rate
- Identify high-risk customer segments
- Analyze churn by contract and internet service
- Evaluate support and security service patterns
- Analyze churn across customer tenure groups
- Translate analytical findings into retention recommendations


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

## 2. Load Dataset

The dataset is loaded using a relative file path so the notebook can run after cloning the GitHub repository.


In [ ]:
df = pd.read_csv("Customer churn.csv")

df.head()

## 3. Dataset Inspection

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.info()

In [ ]:
df.describe()

## 4. Data Quality Assessment

The dataset is checked for missing values, complete-row duplicates, and duplicate customer identifiers.


In [ ]:
print("Standard Missing Values:")
print(df.isnull().sum())

print("\nComplete Duplicate Rows:")
print(df.duplicated().sum())

print("\nDuplicate Customer IDs:")
print(df["customerID"].duplicated().sum())

### TotalCharges Data Cleaning

`TotalCharges` is stored as an object column because blank-string values exist in the source dataset.

The column is converted to numeric using `errors="coerce"` so invalid or blank values become missing values and can be investigated explicitly.


In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing TotalCharges after conversion:")
print(df["TotalCharges"].isna().sum())

df[df["TotalCharges"].isna()]

The missing `TotalCharges` records represent customers with zero tenure. Their total charges are set to 0 because they have not yet accumulated historical charges.


In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Remaining missing values:")
print(df.isnull().sum().sum())

### Senior Citizen Category Cleaning

In [ ]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({
    1: "Yes",
    0: "No"
})

df["SeniorCitizen"].value_counts()

## 5. Overall Customer Churn

In [ ]:
total_customers = len(df)

churned_customers = (
    df["Churn"] == "Yes"
).sum()

retained_customers = (
    df["Churn"] == "No"
).sum()

churn_rate = (
    churned_customers / total_customers
) * 100

print(f"Total Customers: {total_customers:,}")
print(f"Churned Customers: {churned_customers:,}")
print(f"Retained Customers: {retained_customers:,}")
print(f"Overall Churn Rate: {churn_rate:.2f}%")

In [ ]:
ax = sns.countplot(
    data=df,
    x="Churn"
)

ax.bar_label(ax.containers[0])

plt.title("Overall Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.show()

### Business Finding

Approximately 26.54% of customers churned. This means roughly one in four customers left the telecom provider, highlighting a significant customer retention opportunity.


## 6. Reusable Churn Rate Function

A reusable function is created to calculate churn percentages across customer segments.


In [ ]:
def churn_rate_by(column):

    churn_rate = (
        df.groupby(column, observed=True)["Churn"]
        .apply(
            lambda x: (
                x == "Yes"
            ).mean() * 100
        )
        .round(2)
        .sort_values(ascending=False)
    )

    return churn_rate

## 7. Churn by Contract Type

In [ ]:
contract_churn = churn_rate_by("Contract")

contract_churn

In [ ]:
contract_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()

### Business Finding

Month-to-month customers have the highest churn rate. Customers with one-year and two-year contracts demonstrate significantly stronger retention.

**Recommendation:** Prioritize month-to-month customers for loyalty campaigns and contract upgrade incentives.


## 8. Churn by Internet Service

In [ ]:
internet_churn = churn_rate_by("InternetService")

internet_churn

In [ ]:
internet_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()

### Business Finding

Fiber optic customers show the highest churn rate among internet service categories.

**Recommendation:** Investigate fiber optic pricing, service reliability, customer expectations, and technical support interactions.


## 9. Churn by Online Security

In [ ]:
churn_rate_by("OnlineSecurity")

### Business Finding

Customers without Online Security demonstrate higher churn than customers subscribed to the service.

Online Security adoption can be used as a segmentation signal when identifying high-risk customers.


## 10. Churn by Technical Support

In [ ]:
churn_rate_by("TechSupport")

### Business Finding

Customers without Technical Support demonstrate substantially higher churn.

**Recommendation:** Proactive technical support programs may improve customer retention.


## 11. Telecom Service Attribute Analysis

In [ ]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for column in service_columns:

    print(f"\nChurn Rate by {column}")

    print(
        churn_rate_by(column)
    )

## 12. Customer Tenure Segmentation

Customer tenure is grouped into lifecycle segments to compare churn risk across customer maturity levels.


In [ ]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

tenure_churn = churn_rate_by("TenureGroup")

tenure_churn

In [ ]:
tenure_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Customer Tenure")
plt.xlabel("Customer Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()

### Business Finding

Short-tenure customers demonstrate greater churn vulnerability.

The first year of the customer lifecycle represents a critical retention period.

**Recommendation:** Implement stronger onboarding, proactive support, and early engagement programs for newly acquired customers.


## 13. Payment Method Analysis

In [ ]:
payment_churn = churn_rate_by("PaymentMethod")

payment_churn

In [ ]:
payment_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()

## 14. Key Churn Indicators Identified

The exploratory analysis identified the following important churn segmentation factors:

1. Month-to-month contracts
2. Fiber optic internet service
3. Low customer tenure
4. Absence of Online Security
5. Absence of Technical Support
6. Service bundle and support gaps

These findings represent associations observed during exploratory analysis and should not be interpreted as causal relationships.


## 15. Business Recommendations

Based on the exploratory analysis:

- Prioritize month-to-month customers in retention campaigns.
- Encourage annual contract adoption through loyalty incentives.
- Investigate the elevated churn among fiber optic customers.
- Develop an early-tenure customer retention program.
- Provide proactive technical support to high-risk customers.
- Promote relevant Online Security and support service bundles.
- Use the identified churn indicators as candidate features for a future churn prediction model.


## Conclusion

The analysis demonstrates that contract type, internet service, customer tenure, Online Security, and Technical Support are important churn segmentation factors.

The strongest retention opportunity is to focus on short-tenure, month-to-month customers and investigate the elevated churn observed among fiber optic subscribers.

This project demonstrates how exploratory data analysis can translate customer-level telecom data into actionable customer retention insights.
